In [ ]:
# Mount Google Drive
from google.colab import drive  # access Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# Mount Google Drive
from google.colab import drive  # access drive
drive.mount('/content/drive')

# pandas for data handling
import pandas as pd
import numpy as np

# path
path = "/content/drive/MyDrive/BigData/assignment3/ml-latest-small/"

# load data
ratings = pd.read_csv(path + "ratings.csv")
movies = pd.read_csv(path + "movies.csv")

ratings.head()

Mounted at /content/drive


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
# reward: >=4 positive, <4 negative
def get_reward(user_id, movie_id):

    row = ratings[
        (ratings["userId"] == user_id) &
        (ratings["movieId"] == movie_id)
    ]

    if len(row) == 0:
        return 0   # neutral

    rating = row.iloc[0]["rating"]

    if rating >= 4:
        return 1
    else:
        return -1

In [3]:
# reward: >=4 positive, <4 negative
def get_reward(user_id, movie_id):

    row = ratings[
        (ratings["userId"] == user_id) &
        (ratings["movieId"] == movie_id)
    ]

    if len(row) == 0:
        return 0   # neutral

    rating = row.iloc[0]["rating"]

    if rating >= 4:
        return 1
    else:
        return -1

In [4]:
# parameters
epsilon = 0.1   # exploration
num_movies = ratings["movieId"].nunique()

movie_ids = ratings["movieId"].unique()

# initialize reward estimates
Q = {m: 0.0 for m in movie_ids}   # estimated reward
N = {m: 0 for m in movie_ids}     # count

def choose_movie_epsilon_greedy():

    if np.random.rand() < epsilon:
        # explore
        return np.random.choice(movie_ids)
    else:
        # exploit
        return max(Q, key=Q.get)

def update_q(movie_id, reward):

    N[movie_id] += 1
    Q[movie_id] += (reward - Q[movie_id]) / N[movie_id]

In [5]:
# simulate for one user
def run_mab(user_id, steps=500):

    rewards = []

    for _ in range(steps):

        movie = choose_movie_epsilon_greedy()
        reward = get_reward(user_id, movie)

        update_q(movie, reward)
        rewards.append(reward)

    return rewards

In [6]:
rewards = run_mab(user_id=1, steps=500)

print("Average Reward:", np.mean(rewards))

Average Reward: 0.91


In [7]:
# parameters
alpha = 0.1    # learning rate
gamma = 0.9    # discount
epsilon = 0.1

users = ratings["userId"].unique()
movies_list = ratings["movieId"].unique()

# initialize Q-table
Q_table = {}

for u in users:
    for m in movies_list:
        Q_table[(u, m)] = 0.0

In [8]:
def choose_action_q(user_id):

    if np.random.rand() < epsilon:
        return np.random.choice(movies_list)
    else:
        # best movie for user
        values = [Q_table[(user_id, m)] for m in movies_list]
        return movies_list[np.argmax(values)]

In [9]:
def update_q_learning(user_id, movie_id, reward):

    current_q = Q_table[(user_id, movie_id)]

    # max future Q
    future_q = max([Q_table[(user_id, m)] for m in movies_list])

    # update rule
    Q_table[(user_id, movie_id)] = current_q + alpha * (
        reward + gamma * future_q - current_q
    )

In [10]:
def run_q_learning(steps=1000):

    rewards = []

    for _ in range(steps):

        user = np.random.choice(users)
        movie = choose_action_q(user)

        reward = get_reward(user, movie)

        update_q_learning(user, movie, reward)

        rewards.append(reward)

    return rewards

In [11]:
q_rewards = run_q_learning(steps=1000)

print("Average Reward (Q-Learning):", np.mean(q_rewards))

Average Reward (Q-Learning): 0.168


In [12]:
def recommend_rl(user_id, top_k=10):

    scores = [(m, Q_table[(user_id, m)]) for m in movies_list]

    scores.sort(key=lambda x: x[1], reverse=True)

    top_movies = [m[0] for m in scores[:top_k]]

    return movies[movies["movieId"].isin(top_movies)][["movieId", "title"]]

In [13]:
recommend_rl(1, top_k=10)

,movieId,title
0,1,Toy Story (1995)
2,3,Grumpier Old Men (1995)
5,6,Heat (1995)
43,47,Seven (a.k.a. Se7en) (1995)
46,50,"Usual Suspects, The (1995)"
62,70,From Dusk Till Dawn (1996)
89,101,Bottle Rocket (1996)
97,110,Braveheart (1995)
124,151,Rob Roy (1995)
130,157,Canadian Bacon (1995)


In [14]:
explore_count = 0
exploit_count = 0

for _ in range(500):
    if np.random.rand() < epsilon:
        explore_count += 1
    else:
        exploit_count += 1

print("Exploration:", explore_count)
print("Exploitation:", exploit_count)

Exploration: 39
Exploitation: 461


In [15]:
print("RL Avg Reward:", np.mean(q_rewards))
print("MAB Avg Reward:", np.mean(rewards))

RL Avg Reward: 0.168
MAB Avg Reward: 0.91
